In [1]:
"""
Test whether the bat_score -> exit-intention relationship is MODERATED by
crisis intensity (rolling 14-day severe-CVE count from NVD, computed in
build_crisis_intensity_post_level.py).

Model: logistic regression
  has_exit_intent ~ bat_score + crisis_intensity + bat_score:crisis_intensity
                     + days_since_start (linear time control)

The interaction term (bat_score:crisis_intensity) is the actual answer to
"does the relationship change depending on crisis proximity":
  - significant + positive: burnout severity predicts exit intent MORE
    strongly during high-crisis-intensity periods
  - significant + negative: the relationship WEAKENS during high-intensity
    periods
  - not significant: no evidence the relationship changes at all

Run separately for crisis_intensity_run1 (>9.5) and crisis_intensity_run2
(==10).
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/q3_analysis/"
POST_LEVEL_CSV = os.path.join(OUT_DIR, "crisis_intensity_post_level.csv")

ALPHA = 0.05


def run_interaction_model(df, intensity_col, label):
    print(f"\n{'=' * 70}")
    print(f"MODEL: has_exit_intent ~ bat_score + {intensity_col} + "
          f"bat_score:{intensity_col} + days_since_start")
    print("=" * 70)

    # Standardize the continuous predictors so the interaction coefficient
    # is on an interpretable, comparable scale (otherwise raw bat_score
    # 0-4 vs crisis_intensity in the hundreds makes the interaction
    # coefficient's magnitude hard to read).
    d = df.copy()
    for col in ["bat_score", intensity_col, "days_since_start"]:
        d[f"{col}_z"] = (d[col] - d[col].mean()) / d[col].std()

    formula = (f"has_exit_intent ~ bat_score_z + {intensity_col}_z + "
               f"bat_score_z:{intensity_col}_z + days_since_start_z")

    model = smf.logit(formula, data=d).fit(disp=0)
    print(model.summary2().tables[1].to_string())

    interaction_term = f"bat_score_z:{intensity_col}_z"
    coef = model.params.get(interaction_term, np.nan)
    pvalue = model.pvalues.get(interaction_term, np.nan)
    sig = "SIGNIFICANT" if pvalue < ALPHA else "not significant"

    direction = "strengthens" if coef > 0 else "weakens"
    print(f"\n  Interaction term ({interaction_term}): coef={coef:.4f}, p={pvalue:.4g} -> {sig}")
    if pvalue < ALPHA:
        print(f"  Interpretation: the bat_score -> exit-intent relationship {direction} "
              f"as {label} crisis intensity increases.")
    else:
        print(f"  Interpretation: no evidence the relationship changes with {label} crisis intensity.")

    return {
        "intensity_measure": label,
        "interaction_coef": round(coef, 4),
        "interaction_p": round(pvalue, 6),
        "significant": pvalue < ALPHA,
        "n_obs": int(model.nobs),
    }


def main():
    if not os.path.exists(POST_LEVEL_CSV):
        print(f"Could not find {POST_LEVEL_CSV}. Run build_crisis_intensity_post_level.py first.")
        return

    df = pd.read_csv(POST_LEVEL_CSV)
    print(f"Loaded {len(df)} posts from {POST_LEVEL_CSV}")
    print(f"has_exit_intent rate: {df['has_exit_intent'].mean():.4f}")

    results = []
    results.append(run_interaction_model(df, "crisis_intensity_run1", "Run1 (>9.5)"))
    results.append(run_interaction_model(df, "crisis_intensity_run2", "Run2 (==10)"))

    results_df = pd.DataFrame(results)
    out_path = os.path.join(OUT_DIR, "crisis_moderation_results.csv")
    results_df.to_csv(out_path, index=False)

    print(f"\n{'=' * 70}")
    print("SUMMARY")
    print("=" * 70)
    print(results_df.to_string(index=False))
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()

Loaded 135897 posts from /Users/nadia/Desktop/redditRun_june/q3_analysis/crisis_intensity_post_level.csv
has_exit_intent rate: 0.0752

MODEL: has_exit_intent ~ bat_score + crisis_intensity_run1 + bat_score:crisis_intensity_run1 + days_since_start
                                        Coef.  Std.Err.           z         P>|z|    [0.025    0.975]
Intercept                           -2.976442  0.013665 -217.807072  0.000000e+00 -3.003226 -2.949658
bat_score_z                          0.925461  0.007723  119.828488  0.000000e+00  0.910324  0.940599
crisis_intensity_run1_z              0.045747  0.014338    3.190528  1.420129e-03  0.017644  0.073849
bat_score_z:crisis_intensity_run1_z  0.002494  0.007671    0.325076  7.451235e-01 -0.012542  0.017529
days_since_start_z                   0.217621  0.013340   16.313182  7.953946e-60  0.191474  0.243767

  Interaction term (bat_score_z:crisis_intensity_run1_z): coef=0.0025, p=0.7451 -> not significant
  Interpretation: no evidence the relatio